In [10]:
import pandas as pd
movies_df = pd.read_csv('../../data/movies.csv')
print(movies_df.info())
print(movies_df.head())
print(movies_df.describe(include='all'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941597 entries, 0 to 941596
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id           941597 non-null  int64  
 1   name         941587 non-null  object 
 2   date         849684 non-null  float64
 3   tagline      139387 non-null  object 
 4   description  780785 non-null  object 
 5   minute       760027 non-null  float64
 6   rating       90999 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 50.3+ MB
None
        id                               name    date  \
0  1000001                             Barbie  2023.0   
1  1000002                           Parasite  2019.0   
2  1000003  Everything Everywhere All at Once  2022.0   
3  1000004                         Fight Club  1999.0   
4  1000005                         La La Land  2016.0   

                                            tagline  \
0                  She's everything. 

## Dataset Overview
The Dataset contains general information about movies.
The data is divided in **7 columns**:
- **id**: A unique identifier for each movie
- **name**: The title of the movie
- **date**: The year the movie was released
- **tagline**: The tagline of the movie
- **description**: The plot of the movie
- **minute**: The duration of the movie in minutes
- **rating**: The rating of the movie

---
## Starting the data cleaning process
The goals are:
- **Rename columns**: to give more unique identifiable meaning
- **Handling duplicates and redundant data**
- **Handling logical inconsistency**
- **Look for wrong values**: we have to be sure that the dataset does not contain outliers, wrong or unrealistic values.
- **Casting column value types to more appropriate types**

### 1. **Renaming columns**

In [11]:
movies_df.rename(columns={
    'name':'title',
    'date':'global_release_year',
    'description':'plot',
    'minute':'runtime',
    'rating':'critique_rating',
}, inplace=True)
movies_df.head()

,id,title,global_release_year,tagline,plot,runtime,critique_rating
0,1000001,Barbie,2023.0,She's everything. He's just Ken.,Barbie and Ken are having the time of their li...,114.0,3.86
1,1000002,Parasite,2019.0,Act like you own the place.,"All unemployed, Ki-taek's family takes peculia...",133.0,4.56
2,1000003,Everything Everywhere All at Once,2022.0,The universe is so much bigger than you realize.,An aging Chinese immigrant is swept up in an i...,140.0,4.30
3,1000004,Fight Club,1999.0,Mischief. Mayhem. Soap.,A ticking-time-bomb insomniac and a slippery s...,139.0,4.27
4,1000005,La La Land,2016.0,Here's to the fools who dream.,"Mia, an aspiring actress, serves lattes to mov...",129.0,4.09


### 2. Checking duplicates and redundant data
####    1. Checking general duplicates

In [12]:
duplicated_rows = movies_df.duplicated().sum()
print(f"Duplicated rows in DataFrame:\n{duplicated_rows}")

Duplicated rows in DataFrame:
0


#### 2. Checking duplicates for the *id* column

In [13]:
print(movies_df['id'].duplicated().sum())

0


### 3. Handlig logical inconsistency && Wrong values
#### 1. Checking *global_release_year* column values

In [14]:
print("Nulls in 'global_release_year' before to_numeric:", movies_df['global_release_year'].isnull().sum())

# Convert 'global_release_year' to integer, handling potential errors
movies_df['global_release_year'] = pd.to_numeric(movies_df['global_release_year'], errors='coerce')

# Check for nulls introduced by coercion (if any non-numeric values existed)
print("Nulls in 'global_release_year' after to_numeric:", movies_df['global_release_year'].isnull().sum())

# Check for unrealistic years (assuming movies are not made after 2031 year of announced release of avatar 6)
unrealistic_years = movies_df[movies_df['global_release_year'] > 2031][['id', 'title', 'global_release_year']]
print(f"Movies with unrealistic release years:\n{unrealistic_years[['title', 'global_release_year']]}")

Nulls in 'global_release_year' before to_numeric: 91913
Nulls in 'global_release_year' after to_numeric: 91913
Movies with unrealistic release years:
Empty DataFrame
Columns: [title, global_release_year]
Index: []


#### 2. Checking *critique_rating* column values

In [15]:
print(movies_df['critique_rating'].info())
print(movies_df['critique_rating'].describe(include='all'))
print(f"Nulls in 'critique_rating'", movies_df['critique_rating'].isnull().sum())

<class 'pandas.core.series.Series'>
RangeIndex: 941597 entries, 0 to 941596
Series name: critique_rating
Non-Null Count  Dtype  
--------------  -----  
90999 non-null  float64
dtypes: float64(1)
memory usage: 7.2 MB
None
count    90999.000000
mean         3.244043
std          0.417281
min          0.880000
25%          3.020000
50%          3.300000
75%          3.510000
max          4.690000
Name: critique_rating, dtype: float64
Nulls in 'critique_rating' 850598


### 4. Changing *type* column entries values

In [16]:
movies_df['runtime'] = pd.to_numeric(movies_df['runtime'], errors='coerce').astype('Int64')
movies_df['critique_rating'] = pd.to_numeric(movies_df['critique_rating'], errors='coerce')
movies_df['global_release_year'] = movies_df['global_release_year'].astype('Int64')
movies_df['title'] = movies_df['title'].astype('string')
movies_df['tagline'] = movies_df['tagline'].astype('string')
movies_df['plot'] = movies_df['plot'].astype('string')

print(movies_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941597 entries, 0 to 941596
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   941597 non-null  int64  
 1   title                941587 non-null  string 
 2   global_release_year  849684 non-null  Int64  
 3   tagline              139387 non-null  string 
 4   plot                 780785 non-null  string 
 5   runtime              760027 non-null  Int64  
 6   critique_rating      90999 non-null   float64
dtypes: Int64(2), float64(1), int64(1), string(3)
memory usage: 52.1 MB
None


## Saving the cleaned dataset

In [17]:
movies_df.to_csv('../../data/data-cleaned/movies_cleaned.csv', index=False)
print("Cleaned dataset saved as 'movies_cleaned.csv'")

Cleaned dataset saved as 'movies_cleaned.csv'


In [18]:
%reset -f